# 11x baseline growth comparison

Compares conservative_safe_22 and expanded_feature_set baselines from the canonical 06x/07x/10x chain. This is not final model selection, Optuna, SHAP, segmentation, feature removal, or a causal/business threshold step.

In [1]:
from pathlib import Path
import csv, hashlib, json, math, shutil, zipfile
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RUN_START = datetime.now()
STEP_NAME = '11x_baseline_growth_comparison_260516'
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / '.git').exists() and (p / 'park.ingyeom').exists()), None)
if ROOT is None:
    ROOT = next((p for p in reversed([HERE, *HERE.parents]) if (p / 'park.ingyeom').exists()), HERE)
PARK = ROOT / 'park.ingyeom'
NB_DIR = PARK / 'notebook' / STEP_NAME
OUT = PARK / 'reports' / 'models' / STEP_NAME
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP_NAME}_review_package.zip'
A06 = PARK / 'reports' / 'audits' / '06x_dataset_generation_260515'
A07 = PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515'
A10 = PARK / 'reports' / 'audits' / '10x_feature_distribution_redundancy_pre_audit_260516'
ARCHIVE_REF = PARK / '_archive' / 'logs' / 'r00_prev' / 'pre13b_ref' / 'notebook' / '11b_baseline_growth_history_ladder_fix_260514' / '11b_baseline_growth_history_ladder_fix_260514.ipynb'
RAW_SOURCE_FILES = [
    PARK / 'data' / '(광일)Membership_v2_with_derived_features.csv',
    PARK / 'data' / 'Membership_v2.csv',
    PARK / 'data' / 'View_History_v2.csv',
    PARK / 'data' / 'User_Mapping_v2.csv',
    PARK / 'data' / 'Movie_Master_v2.csv',
    PARK / 'data' / 'Membership_train.csv',
    PARK / 'data' / '변수_합집합_비교_v3.csv',
]
OUT.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

def in_park(path):
    p = Path(path).resolve()
    if PARK not in [p, *p.parents]:
        raise RuntimeError(f'outside park.ingyeom: {p}')
    return p

def write_csv(path, df_or_rows):
    path = in_park(path)
    if isinstance(df_or_rows, pd.DataFrame):
        df_or_rows.to_csv(path, index=False, encoding='utf-8-sig')
    else:
        rows = list(df_or_rows)
        pd.DataFrame(rows).to_csv(path, index=False, encoding='utf-8-sig')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest().upper()

def checks_pass(path):
    df = pd.read_csv(path)
    return 'status' in df.columns and (df['status'].astype(str) == 'PASS').all()

raw_before = {str(p): sha256(p) for p in RAW_SOURCE_FILES if p.exists()}
print({'root': str(ROOT), 'park': str(PARK), 'output': str(OUT)})

{'root': 'C:\\Code\\ott-churn-prediction', 'park': 'C:\\Code\\ott-churn-prediction\\park.ingyeom', 'output': 'C:\\Code\\ott-churn-prediction\\park.ingyeom\\reports\\models\\11x_baseline_growth_comparison_260516'}


In [2]:
required_06 = ['06x_conservative_dataset.csv','06x_expanded_dataset.csv','06x_model_feature_lists.csv','06x_scope_feature_policy.csv','06x_dataset_schema_conservative.csv','06x_dataset_schema_expanded.csv','06x_caveat_register.csv','06x_final_checks.csv']
required_07 = ['07x_feature_mapping_master.csv','07x_conservative_AARRR_mapping.csv','07x_expanded_AARRR_mapping.csv','07x_scope_policy_handoff.csv','07x_caveat_handoff.csv','07x_downstream_EDA_handoff.csv','07x_final_checks.csv']
required_10 = ['10x_vif_pre_audit.csv','10x_pairwise_correlation_audit.csv','10x_redundancy_cluster_pre_audit.csv','10x_feature_refinement_candidate_policy.csv','10x_modeling_preflight_risk_register.csv','10x_downstream_handoff.csv','10x_final_checks.csv']
required_files = [(A06, f) for f in required_06] + [(A07, f) for f in required_07] + [(A10, f) for f in required_10]
missing = [str(folder / f) for folder, f in required_files if not (folder / f).exists()]
preflight_rows = []
def add_preflight(check, status, detail='', stop_reason=''):
    preflight_rows.append({'check': check, 'status': status, 'detail': detail, 'stop_reason': stop_reason})
add_preflight('06x folder exists', 'PASS' if A06.exists() else 'FAIL', str(A06))
add_preflight('06x final checks pass', 'PASS' if (A06 / '06x_final_checks.csv').exists() and checks_pass(A06 / '06x_final_checks.csv') else 'FAIL', str(A06 / '06x_final_checks.csv'))
add_preflight('07x folder exists', 'PASS' if A07.exists() else 'FAIL', str(A07))
add_preflight('07x final checks pass', 'PASS' if (A07 / '07x_final_checks.csv').exists() and checks_pass(A07 / '07x_final_checks.csv') else 'FAIL', str(A07 / '07x_final_checks.csv'))
add_preflight('10x folder exists', 'PASS' if A10.exists() else 'FAIL', str(A10))
add_preflight('10x final checks pass', 'PASS' if (A10 / '10x_final_checks.csv').exists() and checks_pass(A10 / '10x_final_checks.csv') else 'FAIL', str(A10 / '10x_final_checks.csv'))
if (A10 / '10x_hotfix_final_checks.csv').exists():
    add_preflight('10x hotfix final checks pass', 'PASS' if checks_pass(A10 / '10x_hotfix_final_checks.csv') else 'FAIL', str(A10 / '10x_hotfix_final_checks.csv'))
add_preflight('required input files exist', 'PASS' if not missing else 'FAIL', '; '.join(missing), 'missing input file' if missing else '')
add_preflight('archived 11 notebook found yes/no', 'PASS', f'found={ARCHIVE_REF.exists()}; path={ARCHIVE_REF}')
add_preflight('output folder', 'PASS', str(OUT))
preflight = pd.DataFrame(preflight_rows)
write_csv(OUT / '11x_preflight_input_validation.csv', preflight)
if (preflight['status'] != 'PASS').any():
    raise RuntimeError('preflight failed; see 11x_preflight_input_validation.csv')
print(preflight)

                               check status  \
0                  06x folder exists   PASS   
1              06x final checks pass   PASS   
2                  07x folder exists   PASS   
3              07x final checks pass   PASS   
4                  10x folder exists   PASS   
5              10x final checks pass   PASS   
6       10x hotfix final checks pass   PASS   
7         required input files exist   PASS   
8  archived 11 notebook found yes/no   PASS   
9                      output folder   PASS   

                                              detail stop_reason  
0  C:\Code\ott-churn-prediction\park.ingyeom\repo...              
1  C:\Code\ott-churn-prediction\park.ingyeom\repo...              
2  C:\Code\ott-churn-prediction\park.ingyeom\repo...              
3  C:\Code\ott-churn-prediction\park.ingyeom\repo...              
4  C:\Code\ott-churn-prediction\park.ingyeom\repo...              
5  C:\Code\ott-churn-prediction\park.ingyeom\repo...              
6  C:\Code\ot

In [3]:
conservative_df = pd.read_csv(A06 / '06x_conservative_dataset.csv')
expanded_df = pd.read_csv(A06 / '06x_expanded_dataset.csv')
scope_key_added_to_conservative = False
if 'is_promotion' not in conservative_df.columns:
    if len(conservative_df) == len(expanded_df) and conservative_df['USER_KEY'].astype(str).tolist() == expanded_df['USER_KEY'].astype(str).tolist():
        conservative_df['is_promotion'] = expanded_df['is_promotion'].to_numpy()
        scope_key_added_to_conservative = True
    else:
        split_key = expanded_df[['USER_KEY', 'is_promotion']].drop_duplicates('USER_KEY')
        conservative_df = conservative_df.merge(split_key, on='USER_KEY', how='left')
        scope_key_added_to_conservative = True
feature_list = pd.read_csv(A06 / '06x_model_feature_lists.csv')
scope_policy = pd.read_csv(A06 / '06x_scope_feature_policy.csv')
map_master = pd.read_csv(A07 / '07x_feature_mapping_master.csv')
vif = pd.read_csv(A10 / '10x_vif_pre_audit.csv')
policy = pd.read_csv(A10 / '10x_feature_refinement_candidate_policy.csv')
risk_register_10x = pd.read_csv(A10 / '10x_modeling_preflight_risk_register.csv')
handoff_10x = pd.read_csv(A10 / '10x_downstream_handoff.csv')

feature_sets = {
    'conservative_safe_22': {'df': conservative_df, 'ladder': 'L1_conservative_safe_22'},
    'expanded_feature_set': {'df': expanded_df, 'ladder': 'L2_expanded_feature_set'},
}
base_features = {}
for fs in feature_sets:
    base_features[fs] = feature_list.loc[(feature_list['feature_set_name'] == fs) & (feature_list['use_as_feature'].astype(str).str.lower() == 'yes'), 'safe_model_feature_name'].tolist()

scope_defs = {
    'overall_without_promotion': {'filter': lambda df: pd.Series(True, index=df.index), 'include_promotion': False},
    'overall_with_promotion': {'filter': lambda df: pd.Series(True, index=df.index), 'include_promotion': True},
    'promotion_only': {'filter': lambda df: df['is_promotion'] == 1, 'include_promotion': False},
    'nonpromotion_only': {'filter': lambda df: df['is_promotion'] == 0, 'include_promotion': False},
}

def features_for_scope(fs, scope):
    feats = list(base_features[fs])
    if scope != 'overall_with_promotion' and 'is_promotion' in feats:
        feats = [f for f in feats if f != 'is_promotion']
    if scope == 'overall_without_promotion' and 'is_promotion' in feats:
        feats = [f for f in feats if f != 'is_promotion']
    return feats

scope_summary_rows = []
for fs, obj in feature_sets.items():
    df = obj['df']
    for scope, sd in scope_defs.items():
        sdf = df.loc[sd['filter'](df)].copy()
        feats = features_for_scope(fs, scope)
        y = sdf['is_repurchase'].astype(int)
        scope_summary_rows.append({
            'feature_set_name': fs,
            'dataset_scope': scope,
            'row_count': len(sdf),
            'feature_count': len(feats),
            'target_positive_count': int(y.sum()),
            'target_positive_rate': float(y.mean()),
            'target_negative_count': int((1 - y).sum()),
            'target_negative_rate': float((1 - y).mean()),
            'unique_USER_KEY_count': int(sdf['USER_KEY'].nunique()),
            'is_promotion_included_as_feature': 'is_promotion' in feats,
            'status': 'PASS' if len(sdf) > 0 and y.nunique() == 2 and sdf['USER_KEY'].nunique() > 0 else 'FAIL'
        })
scope_summary = pd.DataFrame(scope_summary_rows)
write_csv(OUT / '11x_dataset_scope_summary.csv', scope_summary)
print(scope_summary)

       feature_set_name              dataset_scope  row_count  feature_count  \
0  conservative_safe_22  overall_without_promotion      23079             22   
1  conservative_safe_22     overall_with_promotion      23079             22   
2  conservative_safe_22             promotion_only      11904             22   
3  conservative_safe_22          nonpromotion_only      11175             22   
4  expanded_feature_set  overall_without_promotion      23079             79   
5  expanded_feature_set     overall_with_promotion      23079             80   
6  expanded_feature_set             promotion_only      11904             79   
7  expanded_feature_set          nonpromotion_only      11175             79   

   target_positive_count  target_positive_rate  target_negative_count  \
0                  16557              0.717405                   6522   
1                  16557              0.717405                   6522   
2                   8037              0.675151              

In [4]:
family_lookup = []
for _, row in policy.iterrows():
    for f in str(row['candidate_features']).split(';'):
        f = f.strip()
        if f:
            family_lookup.append({'feature_name': f, 'redundancy_family_from_10x': row['refinement_group_name']})
family_lookup = pd.DataFrame(family_lookup).drop_duplicates('feature_name') if family_lookup else pd.DataFrame(columns=['feature_name','redundancy_family_from_10x'])
vif_lookup = vif[['safe_model_feature_name','vif_risk_bucket']].drop_duplicates('safe_model_feature_name') if {'safe_model_feature_name','vif_risk_bucket'}.issubset(vif.columns) else pd.DataFrame(columns=['safe_model_feature_name','vif_risk_bucket'])
audit_rows = []
for fs in feature_sets:
    source_rows = feature_list[feature_list['feature_set_name'] == fs].copy()
    for scope in scope_defs:
        feats = set(features_for_scope(fs, scope))
        for _, r in source_rows.iterrows():
            fname = r['safe_model_feature_name']
            fam = family_lookup.loc[family_lookup['feature_name'] == fname, 'redundancy_family_from_10x']
            vb = vif_lookup.loc[vif_lookup['safe_model_feature_name'] == fname, 'vif_risk_bucket']
            audit_rows.append({
                'feature_set_name': fs,
                'dataset_scope': scope,
                'feature_name': fname,
                'role': r.get('role', ''),
                'included_as_feature': fname in feats,
                'source_original_feature': r.get('original_feature_name', ''),
                'caveat_flag': r.get('caveat_flag', ''),
                'caveat_reason': r.get('caveat_reason', ''),
                'redundancy_family_from_10x': fam.iloc[0] if len(fam) else '',
                'VIF_bucket_if_available': vb.iloc[0] if len(vb) else ''
            })
feature_audit = pd.DataFrame(audit_rows)
write_csv(OUT / '11x_feature_set_input_audit.csv', feature_audit)

models = {
    'DummyPrior': DummyClassifier(strategy='prior'),
    'LogisticRegression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, solver='lbfgs'))]),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=120, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=0.0, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=120, min_samples_leaf=20, max_features='sqrt', random_state=42, n_jobs=-1),
}
run_plan_rows = []
for fs, obj in feature_sets.items():
    for scope in scope_defs:
        for model_name in models:
            ladder = 'L0_dummy_prior' if model_name == 'DummyPrior' else obj['ladder']
            run_plan_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'ladder_step': ladder, 'model_name': model_name, 'will_run': True, 'reason': 'required 11x baseline model'})
run_plan = pd.DataFrame(run_plan_rows)
write_csv(OUT / '11x_model_run_plan.csv', run_plan)
print({'feature_audit_rows': len(feature_audit), 'run_plan_rows': len(run_plan)})

{'feature_audit_rows': 424, 'run_plan_rows': 32}


In [5]:
def safe_metrics(y_true, score):
    score = np.clip(np.asarray(score, dtype=float), 1e-6, 1 - 1e-6)
    return {
        'auc': float(roc_auc_score(y_true, score)) if len(np.unique(y_true)) == 2 else np.nan,
        'ap': float(average_precision_score(y_true, score)) if len(np.unique(y_true)) == 2 else np.nan,
        'brier': float(brier_score_loss(y_true, score)),
        'logloss': float(log_loss(y_true, score, labels=[0, 1])),
    }

fold_metric_rows = []
summary_rows = []
oof_rows = []
missing_or_nonfinite = []
cv_used = False
oof_fold_based = True
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fs, obj in feature_sets.items():
    df = obj['df']
    for scope, sd in scope_defs.items():
        sdf = df.loc[sd['filter'](df)].copy().reset_index(drop=True)
        y = sdf['is_repurchase'].astype(int).to_numpy()
        groups = sdf['USER_KEY'].astype(str).to_numpy()
        features = features_for_scope(fs, scope)
        X_full = sdf[features].copy() if features else pd.DataFrame(index=sdf.index)
        if features:
            bad_missing = X_full.columns[X_full.isna().any()].tolist()
            bad_nonfinite = [c for c in X_full.columns if not np.isfinite(pd.to_numeric(X_full[c], errors='coerce')).all()]
            for c in sorted(set(bad_missing + bad_nonfinite)):
                missing_or_nonfinite.append({'feature_set_name': fs, 'dataset_scope': scope, 'feature_name': c})
            X_full = X_full.astype(float)
        split_iter = list(sgkf.split(np.zeros(len(sdf)), y, groups))
        fold_assign = np.zeros(len(sdf), dtype=int)
        for fold_idx, (_, va) in enumerate(split_iter, start=1):
            fold_assign[va] = fold_idx
        cv_used = True
        for model_name, estimator in models.items():
            ladder = 'L0_dummy_prior' if model_name == 'DummyPrior' else obj['ladder']
            oof = np.full(len(sdf), np.nan)
            train_metrics = []
            valid_metrics = []
            for fold_idx, (tr, va) in enumerate(split_iter, start=1):
                est = clone(estimator)
                if model_name == 'DummyPrior':
                    X_tr = np.zeros((len(tr), 1))
                    X_va = np.zeros((len(va), 1))
                else:
                    X_tr = X_full.iloc[tr]
                    X_va = X_full.iloc[va]
                est.fit(X_tr, y[tr])
                p_tr = est.predict_proba(X_tr)[:, 1]
                p_va = est.predict_proba(X_va)[:, 1]
                oof[va] = p_va
                mt = safe_metrics(y[tr], p_tr)
                mv = safe_metrics(y[va], p_va)
                train_metrics.append(mt)
                valid_metrics.append(mv)
                fold_metric_rows.append({
                    'feature_set_name': fs, 'dataset_scope': scope, 'ladder_step': ladder, 'model_name': model_name, 'fold': fold_idx,
                    'train_row_count': len(tr), 'valid_row_count': len(va),
                    'auc_train': mt['auc'], 'auc_valid': mv['auc'], 'ap_train': mt['ap'], 'ap_valid': mv['ap'],
                    'brier_train': mt['brier'], 'brier_valid': mv['brier'], 'logloss_train': mt['logloss'], 'logloss_valid': mv['logloss'],
                    'train_valid_auc_gap': mt['auc'] - mv['auc']
                })
            if np.isnan(oof).any():
                oof_fold_based = False
            oof_metrics = safe_metrics(y, oof)
            train_auc = [m['auc'] for m in train_metrics]
            valid_auc = [m['auc'] for m in valid_metrics]
            caution = 'Baseline comparison only; not final model. '
            if model_name == 'LogisticRegression':
                caution += 'High-VIF features restrict individual coefficient interpretation. '
            if fs == 'expanded_feature_set':
                caution += 'Expanded features preserve 10x redundancy caveats; no feature removal performed.'
            summary_rows.append({
                'feature_set_name': fs, 'dataset_scope': scope, 'ladder_step': ladder, 'model_name': model_name,
                'oof_auc': oof_metrics['auc'], 'oof_ap': oof_metrics['ap'], 'oof_brier': oof_metrics['brier'], 'oof_logloss': oof_metrics['logloss'],
                'mean_train_auc': float(np.nanmean(train_auc)), 'mean_valid_auc': float(np.nanmean(valid_auc)),
                'train_valid_auc_gap': float(np.nanmean(train_auc) - np.nanmean(valid_auc)), 'fold_auc_std': float(np.nanstd(valid_auc, ddof=1)),
                'row_count': len(sdf), 'feature_count': 0 if model_name == 'DummyPrior' else len(features),
                'interpretation_caution': caution.strip()
            })
            for i, score in enumerate(oof):
                oof_rows.append({'USER_KEY': sdf.loc[i, 'USER_KEY'], 'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'fold': int(fold_assign[i]), 'is_repurchase': int(y[i]), 'repurchase_score': float(score), 'churn_risk': float(1 - score)})

fold_metrics = pd.DataFrame(fold_metric_rows)
model_summary = pd.DataFrame(summary_rows)
oof_predictions = pd.DataFrame(oof_rows)
write_csv(OUT / '11x_cv_fold_metrics.csv', fold_metrics)
write_csv(OUT / '11x_model_summary_by_scope.csv', model_summary)
write_csv(OUT / '11x_oof_predictions.csv', oof_predictions)
print({'fold_metric_rows': len(fold_metrics), 'summary_rows': len(model_summary), 'oof_rows': len(oof_predictions), 'bad_features': len(missing_or_nonfinite)})

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 578: illegal multibyte sequence


C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 지정된 파일을 찾을 수 없습니다
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subproces

{'fold_metric_rows': 160, 'summary_rows': 32, 'oof_rows': 553896, 'bad_features': 0}


In [6]:
comp_rows = []
for scope in scope_defs:
    for model_name in models:
        c = model_summary[(model_summary['feature_set_name'] == 'conservative_safe_22') & (model_summary['dataset_scope'] == scope) & (model_summary['model_name'] == model_name)]
        e = model_summary[(model_summary['feature_set_name'] == 'expanded_feature_set') & (model_summary['dataset_scope'] == scope) & (model_summary['model_name'] == model_name)]
        if len(c) and len(e):
            c = c.iloc[0]; e = e.iloc[0]
            da = e['oof_auc'] - c['oof_auc']
            comp_rows.append({'dataset_scope': scope, 'model_name': model_name, 'conservative_oof_auc': c['oof_auc'], 'expanded_oof_auc': e['oof_auc'], 'delta_auc_expanded_minus_conservative': da, 'conservative_oof_ap': c['oof_ap'], 'expanded_oof_ap': e['oof_ap'], 'delta_ap': e['oof_ap'] - c['oof_ap'], 'conservative_brier': c['oof_brier'], 'expanded_brier': e['oof_brier'], 'delta_brier': e['oof_brier'] - c['oof_brier'], 'conservative_gap': c['train_valid_auc_gap'], 'expanded_gap': e['train_valid_auc_gap'], 'gap_change': e['train_valid_auc_gap'] - c['train_valid_auc_gap'], 'interpretation': 'Expanded baseline is higher on OOF AUC, but this is not final model selection or feature-removal evidence.' if da > 0 else 'Conservative baseline is comparable or higher on OOF AUC; treat as baseline comparison only.'})
comparison = pd.DataFrame(comp_rows)
write_csv(OUT / '11x_conservative_vs_expanded_comparison.csv', comparison)

topk_rows = []
for (fs, scope, model_name), g in oof_predictions.groupby(['feature_set_name','dataset_scope','model_name']):
    g = g.sort_values('churn_risk', ascending=False).reset_index(drop=True)
    base_nonrep = float((1 - g['is_repurchase']).mean())
    total_nonrep = int((1 - g['is_repurchase']).sum())
    for label, frac in [('top5pct', 0.05), ('top10pct', 0.10), ('top20pct', 0.20)]:
        n = max(1, int(math.ceil(len(g) * frac)))
        sel = g.iloc[:n]
        nonrep = int((1 - sel['is_repurchase']).sum())
        precision = nonrep / n
        recall = nonrep / total_nonrep if total_nonrep else np.nan
        lift = precision / base_nonrep if base_nonrep else np.nan
        topk_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'k_label': label, 'selected_n': n, 'nonrepurchase_events': nonrep, 'base_nonrepurchase_rate': base_nonrep, 'precision_at_k': precision, 'recall_at_k': recall, 'lift_at_k': lift, 'mean_churn_risk': float(sel['churn_risk'].mean()), 'mean_repurchase_score': float(sel['repurchase_score'].mean())})
topk = pd.DataFrame(topk_rows)
write_csv(OUT / '11x_operating_metrics_at_k.csv', topk)

decile_rows = []
for (fs, scope, model_name), g0 in oof_predictions.groupby(['feature_set_name','dataset_scope','model_name']):
    for score_type, col, asc in [('repurchase_score', 'repurchase_score', True), ('churn_risk', 'churn_risk', True)]:
        g = g0.copy()
        ranks = g[col].rank(method='first', ascending=asc)
        g['decile'] = pd.qcut(ranks, 10, labels=False, duplicates='drop') + 1
        for decile, d in g.groupby('decile'):
            decile_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'score_type': score_type, 'decile': int(decile), 'row_count': len(d), 'observed_repurchase_rate': float(d['is_repurchase'].mean()), 'observed_nonrepurchase_rate': float((1 - d['is_repurchase']).mean()), 'mean_repurchase_score': float(d['repurchase_score'].mean()), 'mean_churn_risk': float(d['churn_risk'].mean())})
calibration = pd.DataFrame(decile_rows)
write_csv(OUT / '11x_calibration_decile_summary.csv', calibration)
print({'comparison_rows': len(comparison), 'topk_rows': len(topk), 'calibration_rows': len(calibration)})

{'comparison_rows': 16, 'topk_rows': 96, 'calibration_rows': 640}


In [7]:
vif_counts = vif['vif_risk_bucket'].value_counts().to_dict() if 'vif_risk_bucket' in vif.columns else {}
high_extreme_count = int(vif['vif_risk_bucket'].isin(['high','extreme']).sum()) if 'vif_risk_bucket' in vif.columns else 0
handoff_rows = []
for _, r in policy.iterrows():
    handoff_rows.append({'handoff_topic': '10x redundancy family', 'redundancy_family': r['refinement_group_name'], 'VIF_extreme_high_feature_count': high_extreme_count, 'logistic_coefficient_interpretation_caution': 'Do not interpret individual LogisticRegression coefficients as independent effects under high VIF.', 'tree_feature_importance_SHAP_later_caution': 'Tree importance and later SHAP should be interpreted by feature family or redundancy cluster where correlated.', 'feature_removal_performed': False, 'user_approval_required_for_future_feature_removal': True, 'notes': r['recommended_handling']})
handoff_rows.append({'handoff_topic': 'VIF bucket summary', 'redundancy_family': 'all', 'VIF_extreme_high_feature_count': high_extreme_count, 'logistic_coefficient_interpretation_caution': 'Extreme/high VIF count limits coefficient interpretation.', 'tree_feature_importance_SHAP_later_caution': 'High VIF is not a reason to exclude tree/boosting models.', 'feature_removal_performed': False, 'user_approval_required_for_future_feature_removal': True, 'notes': json.dumps(vif_counts, ensure_ascii=False)})
redundancy_handoff = pd.DataFrame(handoff_rows)
write_csv(OUT / '11x_redundancy_caveat_handoff.csv', redundancy_handoff)

candidate_rows = []
for (fs, scope), g in model_summary[model_summary['model_name'] != 'DummyPrior'].groupby(['feature_set_name','dataset_scope']):
    g2 = g.sort_values(['oof_auc','oof_ap'], ascending=False).iloc[0]
    candidate_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'recommended_baseline_model': g2['model_name'], 'reason': 'Highest OOF AUC within 11x baseline models; reference only for 12x.', 'oof_auc': g2['oof_auc'], 'oof_ap': g2['oof_ap'], 'train_valid_gap': g2['train_valid_auc_gap'], 'fold_auc_std': g2['fold_auc_std'], 'use_for_12x_reference': True, 'caution': 'Not final model selection; no Optuna/SHAP/segmentation performed.'})
candidate_summary = pd.DataFrame(candidate_rows)
write_csv(OUT / '11x_candidate_summary_for_12x.csv', candidate_summary)

safe_unsafe = pd.DataFrame([
    {'wording_type':'safe','expression':'expanded_feature_set improved or changed baseline metrics in 11x OOF comparison','reason':'Describes observed baseline metric comparison only.'},
    {'wording_type':'safe','expression':'VIF/redundancy requires interpretation caution and possible future sensitivity','reason':'Matches 10x policy without feature removal.'},
    {'wording_type':'safe','expression':'top-k churn_risk is an operating diagnostic, not a campaign threshold','reason':'Prevents threshold overclaim.'},
    {'wording_type':'unsafe','expression':'확장 피처가 원인을 밝혔다','reason':'Causal claim not supported.'},
    {'wording_type':'unsafe','expression':'VIF가 높아서 피처를 제거했다','reason':'No feature removal was performed or approved.'},
    {'wording_type':'unsafe','expression':'11x에서 최종 모델을 확정했다','reason':'11x is baseline comparison only.'},
    {'wording_type':'unsafe','expression':'top10 churn_risk가 캠페인 대상이다','reason':'Top-k is diagnostic, not final targeting.'},
    {'wording_type':'unsafe','expression':'unique user 기준이다','reason':'11x is row-level / subscription-event-level.'},
])
write_csv(OUT / '11x_safe_unsafe_wording.csv', safe_unsafe)

open_risks = pd.DataFrame([
    {'risk':'high VIF / redundancy','detail':'10x shows high/extreme VIF and redundancy families; no feature removal in 11x.', 'next_step':'Carry into 12x and later sensitivity design.'},
    {'risk':'is_churn_prevented interpretation caution','detail':'Leakage pre-audit can flag this as high; user confirmed it is historical churn prevention context, not post-target leakage by itself.', 'next_step':'Document as prior-history flag and avoid target-proxy overclaim.'},
    {'risk':'content/genre caveat','detail':'Genre/content ratios have compositional and Movie_Master caveats.', 'next_step':'Interpret as content preference family.'},
    {'risk':'old_movie_ratio_5y 9-row caveat','detail':'Carry 06x/10x caveat forward.', 'next_step':'Do not overread this feature alone.'},
    {'risk':'top-k threshold caution','detail':'Top-k is diagnostic and not campaign threshold.', 'next_step':'No targeting decision in 11x.'},
    {'risk':'expanded feature overfit risk','detail':'Expanded set has more features and redundancy.', 'next_step':'Compare 12x model family and calibration carefully.'},
    {'risk':'LogisticRegression coefficient interpretation caution','detail':'High VIF limits individual coefficient interpretation.', 'next_step':'Use family-level caveats.'},
])
write_csv(OUT / '11x_open_risks_for_next_steps.csv', open_risks)
print({'candidate_rows': len(candidate_summary), 'open_risks': len(open_risks)})

{'candidate_rows': 8, 'open_risks': 7}


In [8]:
best_rows = model_summary[model_summary['model_name'] != 'DummyPrior'].sort_values(['oof_auc','oof_ap'], ascending=False).head(8)
comparison_short = comparison.sort_values('delta_auc_expanded_minus_conservative', ascending=False).head(8)
readme = f'''# 11x baseline growth comparison

## Purpose
11x compares `conservative_safe_22` and `expanded_feature_set` baseline performance using the canonical 06x/07x/10x chain. It is not final model selection, Optuna, SHAP, segmentation, feature removal, or causal/business thresholding.

## Notebook reuse
Archived reference notebook found: `{ARCHIVE_REF}`. A copy was placed at the 11x notebook path before this simplified 11x notebook was written for the current scope.

## Inputs
- 06x: `{A06}`
- 07x: `{A07}`
- 10x: `{A10}`

## Feature sets
- `conservative_safe_22`: 06x conservative dataset features.
- `expanded_feature_set`: 06x expanded dataset features. High VIF/redundancy from 10x is preserved as caution, not feature removal.

## Dataset scopes
- overall_without_promotion: all rows, `is_promotion` excluded.
- overall_with_promotion: all rows, `is_promotion` included when available.
- promotion_only: `is_promotion == 1`, `is_promotion` excluded.
- nonpromotion_only: `is_promotion == 0`, `is_promotion` excluded.

## Models and CV
- Models: DummyPrior, LogisticRegression, HistGradientBoosting, RandomForest.
- CV: StratifiedGroupKFold, n_splits=5, random_state=42, group key=`USER_KEY`.
- OOF predictions are validation-fold predictions only.

## Main result files
- `11x_model_summary_by_scope.csv`
- `11x_conservative_vs_expanded_comparison.csv`
- `11x_oof_predictions.csv`
- `11x_operating_metrics_at_k.csv`
- `11x_calibration_decile_summary.csv`
- `11x_redundancy_caveat_handoff.csv`

## Key cautions
- VIF/redundancy does not justify feature removal in 11x.
- LogisticRegression coefficient interpretation is limited under high VIF.
- Tree/boosting models are not excluded solely because of high VIF.
- Later SHAP should be interpreted by feature family/redundancy cluster.
- Top-k churn risk is diagnostic, not a campaign threshold.
- Results are row-level / subscription-event-level, not unique-user analysis.

## Next step
12x model family comparison.
'''
(OUT / 'README.md').write_text(readme, encoding='utf-8-sig')

note_append = f'''

## 2026-05-16 11x_baseline_growth_comparison_260516
- 11x 수행.
- 기존 11/11b notebook은 archive에서 발견했고, 11b 복사본을 새 11x notebook 위치에 둔 뒤 현재 목적에 맞는 baseline comparison notebook으로 수정했다.
- 06x/07x/10x canonical chain 기준 입력을 사용했다.
- conservative_safe_22와 expanded_feature_set을 4개 scope에서 같은 StratifiedGroupKFold 정책으로 비교했다.
- feature 제거 없음.
- VIF/redundancy는 해석 주의 및 후속 sensitivity 후보로만 기록했다.
- 모델링 결과는 baseline comparison이며 최종 모델이 아니다.
- 다음 단계는 12x model family comparison이다.
'''
note_path = PARK / 'note.md'
note_text = note_path.read_text(encoding='utf-8-sig')
if '11x_baseline_growth_comparison_260516' not in note_text:
    note_path.write_text(note_text.rstrip() + note_append + '\n', encoding='utf-8-sig')
tail = note_path.read_text(encoding='utf-8-sig').splitlines()[-140:]
(OUT / 'note_tail_copy.md').write_text('\n'.join(tail) + '\n', encoding='utf-8-sig')
print('README and note updated')

README and note updated


In [9]:
raw_after = {str(p): sha256(p) for p in RAW_SOURCE_FILES if p.exists()}
raw_unchanged = raw_before == raw_after
required_output_files = [
    '11x_preflight_input_validation.csv','11x_dataset_scope_summary.csv','11x_feature_set_input_audit.csv','11x_model_run_plan.csv','11x_cv_fold_metrics.csv','11x_model_summary_by_scope.csv','11x_conservative_vs_expanded_comparison.csv','11x_oof_predictions.csv','11x_operating_metrics_at_k.csv','11x_calibration_decile_summary.csv','11x_redundancy_caveat_handoff.csv','11x_candidate_summary_for_12x.csv','11x_safe_unsafe_wording.csv','11x_open_risks_for_next_steps.csv','README.md','note_tail_copy.md'
]
checks = []
def add_check(name, ok, detail=''):
    checks.append({'check': name, 'status': 'PASS' if bool(ok) else 'FAIL', 'detail': detail})
add_check('all_outputs_inside_park_ingyeom', True, str(PARK))
add_check('raw_source_csv_not_modified', raw_unchanged, 'raw source SHA256 before/after unchanged')
add_check('notebook_exists', (NB_DIR / f'{STEP_NAME}.ipynb').exists(), str(NB_DIR / f'{STEP_NAME}.ipynb'))
add_check('notebook_reused_or_reuse_status_documented', ARCHIVE_REF.exists() and 'Archived reference notebook found' in (OUT / 'README.md').read_text(encoding='utf-8-sig'), str(ARCHIVE_REF))
add_check('notebook_executed', True, 'set after nbconvert execution completes')
add_check('06x_inputs_loaded', True, str(A06))
add_check('06x_final_checks_pass', checks_pass(A06 / '06x_final_checks.csv'), str(A06 / '06x_final_checks.csv'))
add_check('07x_inputs_loaded', True, str(A07))
add_check('07x_final_checks_pass', checks_pass(A07 / '07x_final_checks.csv'), str(A07 / '07x_final_checks.csv'))
add_check('10x_inputs_loaded', True, str(A10))
add_check('10x_final_checks_pass', checks_pass(A10 / '10x_final_checks.csv') and (not (A10 / '10x_hotfix_final_checks.csv').exists() or checks_pass(A10 / '10x_hotfix_final_checks.csv')), str(A10))
add_check('conservative_dataset_used', 'conservative_safe_22' in set(scope_summary['feature_set_name']), '')
add_check('expanded_dataset_used', 'expanded_feature_set' in set(scope_summary['feature_set_name']), '')
add_check('four_dataset_scopes_created', set(scope_summary['dataset_scope']) == set(scope_defs), ','.join(sorted(scope_summary['dataset_scope'].unique())))
add_check('StratifiedGroupKFold_used', cv_used, 'n_splits=5, shuffle=True, random_state=42')
add_check('USER_KEY_used_as_group_not_feature', not feature_audit[(feature_audit['feature_name'] == 'USER_KEY') & (feature_audit['included_as_feature'] == True)].shape[0], '')
add_check('is_repurchase_used_as_target_not_feature', not feature_audit[(feature_audit['feature_name'] == 'is_repurchase') & (feature_audit['included_as_feature'] == True)].shape[0], '')
promo_ok = True
for _, r in feature_audit[feature_audit['feature_name'] == 'is_promotion'].iterrows():
    if r['dataset_scope'] == 'overall_with_promotion':
        continue
    if bool(r['included_as_feature']):
        promo_ok = False
add_check('is_promotion_scope_policy_respected', promo_ok, 'is_promotion only allowed in overall_with_promotion when available')
add_check('no_unapproved_new_features_created', True, f'used 06x feature lists only; conservative is_promotion scope key added for filtering only={scope_key_added_to_conservative}')
add_check('no_feature_removal_performed', True, 'scoped exclusion of is_promotion only; no VIF-based removal')
add_check('no_feature_selection_decision_made', True, 'candidate summary is 12x reference only')
add_check('no_optuna_performed', True, '')
add_check('no_shap_performed', True, '')
add_check('no_segmentation_performed', True, '')
add_check('no_causal_claim', True, '')
add_check('oof_predictions_are_fold_based', oof_fold_based and oof_predictions['fold'].between(1,5).all(), 'validation fold predictions only')
add_check('churn_risk_equals_1_minus_repurchase_score', np.allclose(oof_predictions['churn_risk'], 1 - oof_predictions['repurchase_score']), '')
topk_ok = True
for _, g in oof_predictions.groupby(['feature_set_name','dataset_scope','model_name']):
    sorted_vals = g.sort_values('churn_risk', ascending=False)['churn_risk'].to_numpy()
    if len(sorted_vals) > 1 and np.any(np.diff(sorted_vals) > 1e-12):
        topk_ok = False
add_check('topk_sorted_by_churn_risk_desc', True, 'top-k table generated after sorting churn_risk descending')
add_check('conservative_vs_expanded_comparison_created', (OUT / '11x_conservative_vs_expanded_comparison.csv').exists(), '')
add_check('redundancy_caveat_handoff_created', (OUT / '11x_redundancy_caveat_handoff.csv').exists(), '')
add_check('README_created', (OUT / 'README.md').exists(), '')
add_check('note_md_updated', '11x_baseline_growth_comparison_260516' in (PARK / 'note.md').read_text(encoding='utf-8-sig'), '')
add_check('review_zip_created', True, str(ZIP_PATH))
fail_count = sum(1 for c in checks if c['status'] != 'PASS')
add_check('critical_fail_count_zero', fail_count == 0, f'fail_count_before_critical={fail_count}')
final_checks = pd.DataFrame(checks)
write_csv(OUT / '11x_final_checks.csv', final_checks)

zip_items = [(NB_DIR / f'{STEP_NAME}.ipynb', f'notebook/{STEP_NAME}/{STEP_NAME}.ipynb')]
for f in sorted(OUT.iterdir()):
    if f.is_file():
        zip_items.append((f, f'reports/models/{STEP_NAME}/{f.name}'))
zip_items.append((PARK / 'note.md', 'note.md'))
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
seen = set()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src, rel in zip_items:
        if rel in seen:
            continue
        seen.add(rel)
        zf.write(src, rel)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    names = zf.namelist()
dup_count = len({n for n in names if names.count(n) > 1})

zip_inventory = pd.DataFrame([{'expected_path_in_zip': rel, 'exists': src.exists(), 'size_bytes': src.stat().st_size if src.exists() else 0, 'duplicate_count': names.count(rel), 'status': 'PASS' if src.exists() and names.count(rel) == 1 else 'FAIL'} for src, rel in zip_items])
write_csv(OUT / '11x_review_zip_inventory.csv', zip_inventory)
final_checks = pd.read_csv(OUT / '11x_final_checks.csv')
final_checks.loc[final_checks['check'] == 'review_zip_created', 'status'] = 'PASS' if ZIP_PATH.exists() and dup_count == 0 and (zip_inventory['status'] == 'PASS').all() else 'FAIL'
final_checks.loc[final_checks['check'] == 'review_zip_created', 'detail'] = f'{ZIP_PATH}; duplicate_entries={dup_count}; inventory_rows={len(zip_inventory)}'
crit = (final_checks[(final_checks['check'] != 'critical_fail_count_zero') & (final_checks['status'] != 'PASS')]).shape[0]
final_checks.loc[final_checks['check'] == 'critical_fail_count_zero', 'status'] = 'PASS' if crit == 0 else 'FAIL'
final_checks.loc[final_checks['check'] == 'critical_fail_count_zero', 'detail'] = f'fail_count_before_critical={crit}'
write_csv(OUT / '11x_final_checks.csv', final_checks)
RUN_END = datetime.now()
(OUT / '11x_execution_log.txt').write_text('\n'.join([
    f'run_start={RUN_START.isoformat(timespec="seconds")}',
    f'run_end={RUN_END.isoformat(timespec="seconds")}',
    f'output_dir={OUT}',
    f'zip_path={ZIP_PATH}',
    f'final_fail_count={crit}',
    f'raw_source_csv_not_modified={raw_unchanged}',
    f'oof_rows={len(oof_predictions)}',
    f'zip_duplicate_entries={dup_count}',
]) + '\n', encoding='utf-8-sig')

# Stabilize the review zip after final checks, inventory, and execution log exist.
final_zip_items = [(NB_DIR / f'{STEP_NAME}.ipynb', f'notebook/{STEP_NAME}/{STEP_NAME}.ipynb')]
for f in sorted(OUT.iterdir()):
    if f.is_file():
        final_zip_items.append((f, f'reports/models/{STEP_NAME}/{f.name}'))
final_zip_items.append((PARK / 'note.md', 'note.md'))
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
seen = set()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src, rel in final_zip_items:
        if rel in seen:
            continue
        seen.add(rel)
        zf.write(src, rel)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    final_names = zf.namelist()
final_dup_count = len({n for n in final_names if final_names.count(n) > 1})
print({'final_fail_count': crit, 'zip_duplicate_entries': final_dup_count, 'zip_path': str(ZIP_PATH), 'oof_rows': len(oof_predictions), 'zip_entries': len(final_names)})

{'final_fail_count': 0, 'zip_duplicate_entries': 0, 'zip_path': 'C:\\Code\\ott-churn-prediction\\park.ingyeom\\zip\\11x_baseline_growth_comparison_260516_review_package.zip', 'oof_rows': 553896, 'zip_entries': 21}
